# Занятие 2. От RAW к RGB

**Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026**
Лекция-опора: **L2 «Формирование изображения: оптика и сенсор»** · **90 минут в классе** · ведёт ассистент

---

Сегодня собираем **маленький ISP своими руками**: берём фото, превращаем его в то, что
на самом деле видит сенсор — серую мозаику Байера (одно число на пиксель), восстанавливаем
цвет билинейной интерполяцией, сравниваем свой результат с `cv2.cvtColor` по PSNR,
а потом измеряем **шум сенсора** по стеку кадров с веб-камеры и проверяем правило σ ∝ √N.

Заготовка начинается ровно на демо 2.1 и 2.3 лекции: вы это уже видели, теперь — руками.

| Минуты | Что происходит |
|--------|----------------|
| 0–15 | Ассистент разбирает опорный пример: мозаика и демозаик OpenCV |
| 15–65 | Вы делаете **TODO 1–3** |
| 65–80 | Разбор решения |
| 80–90 | **Мост к ДЗ 1 «Фотолаборатория»** (выдаётся на следующей неделе) |

Ничего сдавать не нужно: **зачёт ставится в классе** по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется сразу после занятия.

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt   (docs/setup-guide.md)")

import os, time
import cv2, numpy as np
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём фото

Мозаику Байера мы **имитируем из готового JPEG**: настоящий RAW с веб-камеры не достать,
а для изучения артефактов демозаика имитации достаточно (о пределах имитации —
в вопросе после TODO 2). Источники по порядку предпочтения:

1. `data/my_photo.jpg` — **своё фото** (лучше с мелкими деталями: решётки, текст, ветки);
2. фотография здания ИУ из лекции (лежит в репозитории, в Colab — скачается);
3. синтетическая сцена — работает всегда.

Кадр приводится к ширине 960 px и **чётным** размерам: мозаика 2 × 2 иначе не сложится.

In [ ]:
PHOTO_CANDIDATES = [
    "data/my_photo.jpg",                                                 # 1) своё фото
    "../../lectures/L02-optics-sensor/demo/iu-building.jpg",            # 2) фото из лекции (публичный репозиторий)
    "../../../iu-intro-cv/lectures/L02-optics-sensor/demo/iu-building.jpg",   # то же, если ноутбук открыт из iu-intro-cv-ta
    "data/iu-building.jpg",                                              # 2') уже скачанное
]
PHOTO_URL = ("https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/"
             "lectures/L02-optics-sensor/demo/iu-building.jpg")


def make_scene(w=960, h=640, seed=0):
    """Запасной вариант: сцена с мелкими деталями, если нет ни фото, ни сети."""
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), 190, np.uint8)
    for i in range(18):
        c = tuple(int(v) for v in rng.integers(20, 235, 3))
        x, y = int(rng.integers(0, w - 160)), int(rng.integers(0, h - 160))
        cv2.rectangle(img, (x, y), (x + 150, y + 150), c, -1)
        for k in range(x + 10, x + 150, 12):            # «решётка» из тонких линий
            cv2.line(img, (k, y + 10), (k, y + 140), (255, 255, 255), 1)
    return img


def load_photo(width=960):
    img = None
    for p in PHOTO_CANDIDATES:
        if os.path.exists(p):
            img, src = cio.imread(p), p
            break
    if img is None:
        try:                                            # Colab: репозитория рядом нет — качаем
            import urllib.request
            os.makedirs("data", exist_ok=True)
            urllib.request.urlretrieve(PHOTO_URL, "data/iu-building.jpg")
            img, src = cio.imread("data/iu-building.jpg"), PHOTO_URL
        except Exception as e:
            print("фото не скачалось (%s) — синтетическая сцена" % type(e).__name__)
            img, src = make_scene(), "синтетика"
    h = int(round(img.shape[0] * width / img.shape[1]))
    img = cv2.resize(img, (width, h), interpolation=cv2.INTER_AREA)
    img = img[: h - h % 2, : width - width % 2]         # чётные размеры
    print("источник:", src)
    return img


img = load_photo()
print("кадр", img.shape, img.dtype)
viz.show(img, "исходное фото (то, что выдал чей-то ISP)")

## 2. Опорный пример — разбирает ассистент

Это демо 2.1 лекции. Здесь всё написано, задача — **понять каждую строку**.

Сенсор не видит цвета: над каждым пикселем стоит один фильтр — R, G или B — в шахматке
**RGGB**. Мы имитируем это, оставив в каждом пикселе фото **одно** число из трёх.
Потом просим OpenCV восстановить цвет и смотрим, сколько потеряли.

In [ ]:
def to_mosaic(image):
    """Мозаика Байера RGGB: (0,0)=R, (0,1)=G, (1,0)=G, (1,1)=B. Одно число на пиксель."""
    m = np.zeros(image.shape[:2], np.uint8)
    m[0::2, 0::2] = image[0::2, 0::2, 2]     # R  (в BGR красный — канал 2)
    m[0::2, 1::2] = image[0::2, 1::2, 1]     # G
    m[1::2, 0::2] = image[1::2, 0::2, 1]     # G
    m[1::2, 1::2] = image[1::2, 1::2, 0]     # B
    return m


mosaic = to_mosaic(img)
back = cv2.cvtColor(mosaic, cv2.COLOR_BayerBG2BGR)     # демозаик OpenCV (билинейный)

print("фото   ", img.shape, "— три числа на пиксель")
print("мозаика", mosaic.shape, "— ОДНО число на пиксель, серый uint8")
print("PSNR после мозаики и демозаика: %.1f дБ" % metrics.psnr(img, back))

y0, x0 = img.shape[0] // 3, img.shape[1] // 3            # фрагмент 8 × 8 крупно
zoom = lambda a: cv2.resize(a[y0:y0 + 8, x0:x0 + 8], (240, 240), interpolation=cv2.INTER_NEAREST)
viz.grid({"фото, 8 × 8 px": zoom(img), "мозаика: что видит сенсор": zoom(mosaic),
          "после cvtColor": zoom(back)}, cols=3, size=3.2)

---

## TODO 1 — маски Байера *(≈ 10 минут)*

Напишите `bayer_masks(shape, pattern="RGGB")`, возвращающую **три булевых маски**
`(mask_R, mask_G, mask_B)` формы `shape[:2]`: `True` там, где сенсор измерил этот канал.
`pattern` — четыре буквы ячейки 2 × 2 **по строкам**: `RGGB` значит (0,0)=R, (0,1)=G, (1,0)=G, (1,1)=B.

Маски нужны в TODO 2 — по ним мы будем отличать измеренные значения от выдуманных.

> **Подсказка.** `mask[r::2, c::2] = True` красит каждый второй пиксель, начиная с `(r, c)`.
> Для буквы с номером `i` в `pattern`: `r = i // 2`, `c = i % 2`.

In [ ]:
def bayer_masks(shape, pattern="RGGB"):
    """Три булевых маски (R, G, B): где сенсор измерил этот канал."""
    # >>> SOL TODO 1 · 5–8 строк: словарь из трёх нулевых масок, цикл по буквам pattern, срезы с шагом 2
    h, w = shape[:2]
    masks = {c: np.zeros((h, w), bool) for c in "RGB"}
    for i, c in enumerate(pattern):
        masks[c][i // 2::2, i % 2::2] = True
    return masks["R"], masks["G"], masks["B"]
    # <<< SOL


mask_R, mask_G, mask_B = bayer_masks(mosaic.shape)
n_all = mask_R.astype(int) + mask_G + mask_B

assert mask_R.shape == mosaic.shape and mask_R.dtype == bool, "маска — булев массив формы (h, w)"
assert n_all.min() == 1 and n_all.max() == 1, "в каждом пикселе измерен ровно ОДИН канал"
assert mask_G.sum() == 2 * mask_R.sum() == 2 * mask_B.sum(), "зелёных вдвое больше: RGGB"
assert mask_R[0, 0] and mask_B[1, 1] and mask_G[0, 1] and mask_G[1, 0], "раскладка RGGB: R в (0,0), B в (1,1)"
assert np.array_equal(mosaic[mask_R], img[..., 2][mask_R]), "под маской R в мозаике лежит красный канал фото"
gR, gG, gB = bayer_masks(mosaic.shape, "GRBG")
assert gG[0, 0] and gR[0, 1] and gB[1, 0], "GRBG: буквы идут по строкам — R в (0,1), а не в (1,0)"
print("TODO 1 ✔  · измерено: R %d, G %d, B %d из %d пикселей"
      % (mask_R.sum(), mask_G.sum(), mask_B.sum(), mosaic.size))

# RAW «в цвете масок»: каждый пиксель — один канал
colored = np.zeros_like(img)
colored[..., 2][mask_R] = mosaic[mask_R]
colored[..., 1][mask_G] = mosaic[mask_G]
colored[..., 0][mask_B] = mosaic[mask_B]
viz.grid({"фото": zoom(img), "RAW в цвете фильтров": zoom(colored)}, cols=2, size=3.2)

### Ловушка из лекции: какой код `cvtColor`?

Наша мозаика начинается с **R** в углу `(0, 0)`, а правильный код — `COLOR_Bayer**BG**2BGR`.
Код OpenCV называет цвета **второй строки** шаблона (столбцы 2 и 3), а не угла.
Проверяем измерением, а не верой:

In [ ]:
scores = {}
for code in ("BayerBG2BGR", "BayerGB2BGR", "BayerRG2BGR", "BayerGR2BGR"):
    out = cv2.cvtColor(mosaic, getattr(cv2, "COLOR_" + code))
    scores[code] = metrics.psnr(img, out)
    print("%-12s %5.1f дБ" % (code, scores[code]))
best = max(scores, key=scores.get)
print("лучший код для RGGB-мозаики:", best)
assert scores["BayerBG2BGR"] >= max(scores.values()) - 0.5, "для RGGB правильный код — BayerBG2BGR"

## TODO 2 — демозаик руками *(≈ 15 минут)*

Напишите `demosaic_bilinear(mosaic, masks)` — **билинейный демозаик**: для каждого канала
недостающее значение в пикселе = **среднее по измеренным соседям в окне 3 × 3**, а измеренные
значения остаются как есть. Вход — мозаика `uint8` и кортеж масок `(R, G, B)` из TODO 1;
выход — **BGR** `uint8` той же формы, что фото.

> **Подсказка.** `plane = np.where(mask, mosaic, 0)` — канал с нулями там, где не измерено.
> `cv2.boxFilter(x, -1, (3, 3), normalize=False)` — сумма по окну 3 × 3 вокруг каждого пикселя
> (подробно про такие фильтры — L5). Сумма значений ÷ сумма маски = среднее по измеренным
> соседям. Работайте в `float32`, а не в `uint8`: суммы четырёх значений не влезают в байт.
> И не забудьте: OpenCV ждёт порядок каналов **B, G, R**.

In [ ]:
def demosaic_bilinear(mosaic, masks):
    """Билинейный демозаик: пропуски — среднее измеренных соседей 3 × 3, измеренное не трогаем. → BGR uint8."""
    # >>> SOL TODO 2 · 7–10 строк: цикл по маскам R, G, B; boxFilter для значений и для маски; np.where; stack в BGR
    chans = []
    for m in masks:                                          # порядок R, G, B
        plane = np.where(m, mosaic, 0).astype(np.float32)
        s = cv2.boxFilter(plane, -1, (3, 3), normalize=False)             # сумма измеренных соседей
        n = cv2.boxFilter(m.astype(np.float32), -1, (3, 3), normalize=False)   # сколько их
        chans.append(np.where(m, plane, s / n))
    bgr = np.stack(chans[::-1], axis=-1)                     # R, G, B -> B, G, R
    return np.clip(bgr + 0.5, 0, 255).astype(np.uint8)
    # <<< SOL


masks = (mask_R, mask_G, mask_B)
mine = demosaic_bilinear(mosaic, masks)

assert mine.shape == img.shape and mine.dtype == np.uint8, "ожидается BGR uint8 формы фото"
assert np.array_equal(mine[..., 2][mask_R], mosaic[mask_R]), "измеренные значения (R под маской R) должны остаться как есть"
assert np.array_equal(mine[..., 1][mask_G], mosaic[mask_G]), "измеренные значения G должны остаться как есть"
psnr_mine, psnr_cv = metrics.psnr(img, mine), metrics.psnr(img, back)
assert metrics.psnr(mine, back) > 40, "результат должен совпадать с cvtColor почти везде; перепутаны каналы (BGR/RGB)?"
assert abs(psnr_mine - psnr_cv) < 1.0, "PSNR своего демозаика должен совпасть с OpenCV до 1 дБ"
print("TODO 2 ✔  · PSNR к фото: своя %.1f дБ, OpenCV %.1f дБ · между собой %.0f дБ"
      % (psnr_mine, psnr_cv, min(metrics.psnr(mine, back), 99)))

cy, cx = img.shape[0] // 4, img.shape[1] // 4                # фрагмент с мелкими деталями
crop = (slice(cy, cy + 160), slice(cx, cx + 220))
viz.grid({"фото": img[crop], "своя билинейная": mine[crop], "cvtColor": back[crop]},
         cols=3, size=3.6)

### Измеряем: четыре демозаика и одна мишень

Ваш демозаик, `cvtColor` по умолчанию, вариант `_EA` (edge-aware) и `_VNG` (variable number
of gradients). Три числа на каждый: PSNR к фото, доля пикселей с **ложным цветом**
(разность R − B ушла от оригинала больше чем на 32) и время. Последний столбец — мишень
из демо 2.2: серые линии толщиной 1 px.

In [ ]:
def false_color(out, ref):
    """Доля пикселей, где разность R − B ушла от эталона больше чем на 32 уровня."""
    d, r = out.astype(int), ref.astype(int)
    return 100 * float((np.abs((d[..., 2] - d[..., 0]) - (r[..., 2] - r[..., 0])) > 32).mean())


target = np.full((200, 320), 255, np.uint8)               # серая мишень: R = G = B
target[:, ::4] = 0                                        # чёрные линии в 1 px
target_ref = cv2.cvtColor(target, cv2.COLOR_GRAY2BGR)     # «истина»: сцена серая

METHODS = {
    "своя билинейная":  lambda m: demosaic_bilinear(m, bayer_masks(m.shape)),
    "cvtColor (bilinear)": lambda m: cv2.cvtColor(m, cv2.COLOR_BayerBG2BGR),
    "cvtColor _EA":     lambda m: cv2.cvtColor(m, cv2.COLOR_BayerBG2BGR_EA),
    "cvtColor _VNG":    lambda m: cv2.cvtColor(m, cv2.COLOR_BayerBG2BGR_VNG),
}
table = {}
print("%-20s %8s %12s %8s %14s" % ("метод", "PSNR", "лож. цвет", "время", "мишень 1 px"))
for name, f in METHODS.items():
    t0 = time.perf_counter(); out = f(mosaic); dt = 1000 * (time.perf_counter() - t0)
    row = (metrics.psnr(img, out), false_color(out, img), dt, false_color(f(target), target_ref))
    table[name] = row
    print("%-20s %6.1f дБ %9.1f %% %6.1f мс %10.0f %%" % ((name,) + row))

assert table["cvtColor _VNG"][0] > table["cvtColor (bilinear)"][0], "VNG должен выигрывать у билинейного по PSNR"
assert all(row[3] > 50 for row in table.values()), "на 1-px линиях ложный цвет у всех: информации в мозаике нет"
viz.grid({n: cv2.resize(f(target)[60:120, 100:160], (240, 240), interpolation=cv2.INTER_NEAREST)
          for n, f in METHODS.items()}, cols=4, size=2.6, suptitle="мишень 1 px после демозаика")

### Вопрос, на который отвечаем вслух

Ваша функция даёт **ровно то же**, что `cvtColor` по умолчанию, — до пикселя внутри кадра.
Значит, «демозаик OpenCV» — это те же восемь строк. VNG выигрывает 3–4 дБ и вдвое режет
ложный цвет на фото, но на мишени из 1-px линий **все четыре** метода красят 75 % серых
пикселей — почему никакой алгоритм здесь не поможет?

И второй вопрос: мы имитировали мозаику из JPEG, который **уже прошёл** чей-то демозаик,
баланс белого, гамму и сжатие. Что в нашем измерении от этого честно, а что — нет?

---

## TODO 3 — шум сенсора по стеку кадров *(≈ 20 минут)*

Лекция утверждает: пиксель считает фотоны, поэтому **дисперсия шума равна среднему сигналу**
(σ² = N), и усреднение K кадров снижает σ в √K раз. Проверяем на **своей камере**.

Сначала стек: 32 кадра неподвижной сцены с веб-камеры (первые 16 кадров выбрасываем —
автоэкспозиция ещё «дышит»). Нет камеры — ячейка сделает **синтетический** стек: фото
как карта яркости, фотонный шум — Пуассон, как в демо 2.3. **Сидите неподвижно** и не
загораживайте свет: любое движение в кадре — это не шум, а сигнал.

In [ ]:
SOURCE, N_PHOTONS = "synthetic", 400
try:
    frames = list(cio.video_frames(0, max_frames=48, gray=True))
    if len(frames) < 48:
        raise RuntimeError("камера отдала только %d кадров" % len(frames))
    hh, ww = frames[0].shape
    dy, dx = min(240, hh // 2), min(320, ww // 2)               # центральный фрагмент ≤ 480 × 640
    stack = np.stack(frames[16:])[:, hh // 2 - dy:hh // 2 + dy, ww // 2 - dx:ww // 2 + dx].astype(np.float32)
    SOURCE = "camera"
except Exception as e:
    print("камера недоступна (%s) — синтетический стек: фото + пуассоновский шум" % type(e).__name__)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hh, ww = g.shape
    lin = g[hh // 2 - 240:hh // 2 + 240, ww // 2 - 320:ww // 2 + 320].astype(np.float32) / 255.0
    rng = np.random.default_rng(0)
    stack = rng.poisson(lin * N_PHOTONS, (32,) + lin.shape).astype(np.float32)

print("источник: %s · стек %s · единицы: %s" % (
    SOURCE, stack.shape, "коды 0…255 после ISP камеры" if SOURCE == "camera" else "фотоны, до %d" % N_PHOTONS))
viz.grid({"один кадр": stack[0], "среднее 32 кадров": stack.mean(0),
          "|кадр − среднее| × 8": np.clip(np.abs(stack[0] - stack.mean(0)) * 8, 0, 255)}, cols=3, size=3.6)

Теперь напишите `noise_curve(stack, bins=8)` — **кривую шума** (photon transfer curve):
разбейте пиксели на `bins` корзин по среднему сигналу (среднее по оси кадров) и для каждой
корзины верните пару `(средний сигнал, средняя дисперсия по времени)`. Результат — массив
формы `(k, 2)`, строки по возрастанию сигнала; корзины, в которые попало меньше 200 пикселей,
пропускайте.

> **Подсказка.** `stack.mean(0)` и `stack.var(0)` — среднее и дисперсия **по оси кадров**,
> получаются карты формы `(h, w)`. Границы корзин — `np.linspace(mean.min(), mean.max(), bins + 1)`;
> отбор пикселей корзины — булева маска `(mean >= lo) & (mean < hi)`.

In [ ]:
def noise_curve(stack, bins=8):
    """Кривая шума: (средний сигнал, дисперсия по времени) по корзинам яркости. → массив (k, 2)."""
    # >>> SOL TODO 3 · 7–10 строк: карты mean/var по оси 0, границы корзин, цикл с булевым отбором, пропуск малых корзин
    mean, var = stack.mean(0), stack.var(0)
    edges = np.linspace(mean.min(), mean.max(), bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (mean >= lo) & (mean < hi)
        if sel.sum() >= 200:
            rows.append((float(mean[sel].mean()), float(var[sel].mean())))
    return np.array(rows)
    # <<< SOL


curve = noise_curve(stack)
assert curve.ndim == 2 and curve.shape[1] == 2, "ожидается массив (k, 2)"
assert len(curve) >= 4, "слишком мало корзин с данными — сцена почти одноцветная?"
assert np.isfinite(curve).all(), "в кривой NaN или inf"
assert np.all(np.diff(curve[:, 0]) > 0), "строки должны идти по возрастанию сигнала"
slope = np.polyfit(curve[:, 0], curve[:, 1], 1)[0]
print("   сигнал   дисперсия   σ")
for m, v in curve:
    print("   %6.1f   %8.1f   %5.2f" % (m, v, v ** 0.5))
if SOURCE == "synthetic":
    assert 0.8 < slope < 1.2, "для пуассоновского шума дисперсия ≈ сигнал: наклон должен быть около 1"
    print("TODO 3 ✔  · наклон %.2f — дисперсия равна сигналу, свет действительно приходит квантами" % slope)
else:
    print("TODO 3 ✔  · наклон %.2f; прямой линии здесь и не будет — почему, обсуждаем ниже" % slope)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.plot(curve[:, 0], curve[:, 1], "o-", color="#0070C0", label="ваша камера" if SOURCE == "camera" else "синтетика")
if SOURCE == "synthetic":
    ax.plot(curve[:, 0], curve[:, 0], "--", color="#C55A11", label="дисперсия = сигнал")
ax.set_xlabel("средний сигнал"); ax.set_ylabel("дисперсия по времени"); ax.grid(alpha=.3); ax.legend()
plt.show()

### √K: сколько кадров нужно, чтобы шум упал вдвое

Шум по времени удобно мерить **по паре кадров**: разность двух кадров одной сцены убирает
сцену и оставляет только шум, σ = std(разности) / √2 — так делает индустриальный стандарт
EMVA 1288. Усредним по K кадров и посмотрим, как падает σ: лекция обещает ×2 при K = 4
и ×4 при K = 16.

In [ ]:
def temporal_sigma(stack, k):
    """σ шума по времени для среднего из k кадров: по разности двух независимых средних."""
    a, b = stack[:k].mean(0), stack[k:2 * k].mean(0)
    return float((a - b).std() / np.sqrt(2))


sig = {k: temporal_sigma(stack, k) for k in (1, 4, 16)}
print("K = 1: σ = %.2f   K = 4: σ = %.2f (×%.2f)   K = 16: σ = %.2f (×%.2f)"
      % (sig[1], sig[4], sig[1] / sig[4], sig[16], sig[1] / sig[16]))
if SOURCE == "synthetic":
    assert 3.4 < sig[1] / sig[16] < 4.6, "усреднение 16 кадров должно снижать σ примерно в 4 раза"
    print("правило √K подтверждено: 16 кадров — в 4 раза тише, не в 16")
else:
    print("сравните с √K = 2 и 4: если ваша камера не дотягивает — разговор ниже")

### Вопрос, на который отвечаем вслух

У кого стек с камеры: кривая шума **не прямая** и √K **не сходится** — обычно дисперсия растёт
в тенях, а в светах падает; усреднение 16 кадров даёт ×2 вместо ×4. Это не ошибка кода.
Какие стадии ISP из лекции гнут кривую и где именно (гамма, шумоподавление, JPEG, автоэкспозиция,
мерцание ламп 50 Гц)? На каких данных это измерение было бы честным?

У кого синтетика: наклон ≈ 1 и ×4 ровно. Что изменится, если добавить к пуассоновскому шуму
шум чтения — постоянные 3 е⁻ на кадр? Где на кривой это будет видно?

---

## ⭐ Звёздочка — rolling shutter: измерить наклон

Демо 2.4 лекции гнёт вертикальные столбы: строка `y` читается позже и видит сцену сдвинутой
на `y / 4` px. Ячейка ниже строит такой кадр и **измеряет наклон обратно**: находит левый край
первого столба в каждой строке, проводит прямую и печатает угол — должно выйти 14°.

Настоящая звёздочка — **повторить это на своей камере** (только локально): быстро проведите
перед объективом вертикальный предмет (карандаш, край книги), поймайте кадр, где он наклонён,
найдите его край в каждой строке тем же приёмом и оцените сдвиг на строку. Из него — скорость
чтения сенсора: сколько миллисекунд занимает кадр. Сдаётся до конца следующей недели.

In [ ]:
frame = np.full((400, 600), 235, np.uint8)
for x0 in (100, 250, 400):
    frame[:, x0:x0 + 26] = 40                                 # три вертикальных столба
rs = np.stack([np.roll(frame[y], y // 4) for y in range(400)])   # строка y снята позже: сдвиг y/4

ys = np.arange(0, 400, 8)
xs = np.array([int(np.argmax(rs[y] < 128)) for y in ys])       # левый край первого столба в строке
px_per_row = np.polyfit(ys, xs, 1)[0]
print("сдвиг %.3f px на строку → наклон %.1f° (в демо 2.4 было 14°)"
      % (px_per_row, np.degrees(np.arctan(px_per_row))))
viz.grid({"глобальный затвор": frame, "rolling shutter": rs}, cols=2, size=3.6)

try:                                                           # своя камера: 30 кадров, ищем самый «наклонный»
    cam = list(cio.video_frames(0, max_frames=30, gray=True, resize=(640, 360)))
    print("снято %d кадров — выберите кадр с движущимся предметом и повторите замер" % len(cam))
except Exception as e:
    print("камера недоступна:", type(e).__name__)

---

## Мост к домашнему заданию

**ДЗ 1 «Фотолаборатория», выдача на W3, дедлайн W6.** Прямо сегодняшних функций в нём нет —
но в нём есть три стадии ISP, которые вы теперь понимаете изнутри:

| Сегодня | В ДЗ 1 |
|---------|--------|
| демозаик: 2/3 цветов — интерполяция | хромакей: тонким деталям нельзя верить в цвете — отсюда морфология после `inRange` |
| «сумма по окну ÷ число соседей» | это уже свёртка — L5, ДЗ 2; в ДЗ 1 — сглаживание перед порогом |
| кривая шума и гамма гнут статистику | гамма-коррекция — стадия ISP, которую вы делаете сами; недосвет: почему усиление вытаскивает шум |
| баланс белого «серый мир» (слайд 41) | пункт 1 ДЗ: grey-world / white-patch, гистограммы до и после |
| PSNR как способ сравнить варианты | тот же приём для гаммы, баланса белого и CLAHE на своих пяти фото |

Чего сегодня **не** делали и придётся сделать дома: обход папки снимков, обработка битых
файлов и отчёт. Полное ТЗ — `homeworks/hw1-photolab/README.md`, выдаётся на W3.

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–3 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, почему в TODO 2 измеренные значения остаются, а 2/3 — среднее соседей,
      и что делает `boxFilter` с маской;
- [ ] на вопрос про кривую шума вашей камеры (или про √K на синтетике) есть внятный ответ.

**2 % итоговой оценки**, ставится в классе. Досылать ничего не нужно.